In [2]:
import vcf

print("PyVCF3 works!")

PyVCF3 works!


In [3]:
import pandas as pd
import vcf

In [4]:
vcf_reader = vcf.Reader(
    filename="../data/raw/clinvar/clinvar.vcf.gz"
)

In [5]:
rows = []

for record in vcf_reader:

    rows.append({
        "chromosome": record.CHROM,
        "position": record.POS,
        "reference_allele": record.REF,
        "alternate_allele": ",".join(str(x) for x in record.ALT),

        "gene_name": record.INFO.get("GENEINFO"),

        "clinvar_significance": record.INFO.get("CLNSIG"),

        "review_status": record.INFO.get("CLNREVSTAT")
    })

clinvar_df = pd.DataFrame(rows)

UnicodeDecodeError: 'ascii' codec can't decode byte 0xe2 in position 116: ordinal not in range(128)

In [6]:
import gzip

with gzip.open("../data/raw/clinvar/clinvar.vcf.gz", "rt", encoding="utf-8") as f:
    for i in range(20):
        print(f.readline())

##fileformat=VCFv4.1

##fileDate=2026-05-30

##source=ClinVar

##reference=GRCh38

##ID=<Description="ClinVar Variation ID">

##INFO=<ID=AF_ESP,Number=1,Type=Float,Description="allele frequencies from GO-ESP">

##INFO=<ID=AF_EXAC,Number=1,Type=Float,Description="allele frequencies from ExAC">

##INFO=<ID=AF_TGP,Number=1,Type=Float,Description="allele frequencies from TGP">

##INFO=<ID=ALLELEID,Number=1,Type=Integer,Description="the ClinVar Allele ID">

##INFO=<ID=CLNDN,Number=.,Type=String,Description="ClinVar's preferred disease name for the concept specified by disease identifiers in CLNDISDB">

##INFO=<ID=CLNDNINCL,Number=.,Type=String,Description="For included Variant : ClinVar's preferred disease name for the concept specified by disease identifiers in CLNDISDB">

##INFO=<ID=CLNDISDB,Number=.,Type=String,Description="Tag-value pairs of disease database name and identifier submitted for germline classifications, e.g. OMIM:NNNNNN">

##INFO=<ID=CLNDISDBINCL,Number=.,Type=String,Descr

In [7]:
import gzip

with gzip.open(
    "../data/raw/clinvar/clinvar.vcf.gz",
    "rt",
    encoding="utf-8"
) as f:

    for line in f:
        if line.startswith("#"):
            continue

        fields = line.strip().split("\t")

        chromosome = fields[0]
        position = fields[1]
        ref = fields[3]
        alt = fields[4]

        print(chromosome, position, ref, alt)

        break

1 66926 AG A


In [8]:
def parse_info(info_string):

    info_dict = {}

    for item in info_string.split(";"):

        if "=" in item:
            key, value = item.split("=", 1)
            info_dict[key] = value

    return info_dict

In [9]:
import gzip
import pandas as pd

rows = []

with gzip.open(
    "../data/raw/clinvar/clinvar.vcf.gz",
    "rt",
    encoding="utf-8"
) as f:

    for line in f:

        if line.startswith("#"):
            continue

        fields = line.strip().split("\t")

        info_dict = parse_info(fields[7])

        rows.append({

            "chromosome": fields[0],

            "position": int(fields[1]),

            "reference_allele": fields[3],

            "alternate_allele": fields[4],

            "gene_name": info_dict.get("GENEINFO"),

            "clinvar_significance": info_dict.get("CLNSIG"),

            "review_status": info_dict.get("CLNREVSTAT")

        })

        # temporary limit while testing
        if len(rows) == 1000:
            break

In [10]:
clinvar_df = pd.DataFrame(rows)

clinvar_df.head()

,chromosome,position,reference_allele,alternate_allele,gene_name,clinvar_significance,review_status
0,1,66926,AG,A,OR4F5:79501,Uncertain_significance,"criteria_provided,_single_submitter"
1,1,69134,A,G,OR4F5:79501,Likely_benign,"criteria_provided,_single_submitter"
2,1,69241,C,T,OR4F5:79501,Uncertain_significance,"criteria_provided,_single_submitter"
3,1,69308,A,G,OR4F5:79501,Uncertain_significance,"criteria_provided,_single_submitter"
4,1,69314,T,G,OR4F5:79501,Uncertain_significance,"criteria_provided,_single_submitter"


In [11]:
clinvar_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   chromosome            1000 non-null   str  
 1   position              1000 non-null   int64
 2   reference_allele      1000 non-null   str  
 3   alternate_allele      1000 non-null   str  
 4   gene_name             1000 non-null   str  
 5   clinvar_significance  983 non-null    str  
 6   review_status         983 non-null    str  
dtypes: int64(1), str(6)
memory usage: 54.8 KB


In [12]:
clinvar_df["clinvar_significance"].value_counts()

clinvar_significance
Uncertain_significance                          495
Likely_benign                                   424
Benign                                           52
Conflicting_classifications_of_pathogenicity      9
Benign/Likely_benign                              3
Name: count, dtype: int64

In [13]:
clinvar_df.to_csv(
    "../data/interim/clinvar_tables/clinvar_variants.csv",
    index=False
)